<a href="https://colab.research.google.com/github/Najaf-Ali12/LLM-Hugging-Face/blob/main/Finetuning_a_sentiment_analysis_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# Loading required libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import torch
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import EarlyStoppingCallback

In [2]:
# Loading Datasets
from datasets import load_dataset
dataset = load_dataset("syedkhalid0/Sentiment-Analysis")

README.md:   0%|          | 0.00/4.55k [00:00<?, ?B/s]

train_data.csv:   0%|          | 0.00/6.94M [00:00<?, ?B/s]

val_data.csv:   0%|          | 0.00/870k [00:00<?, ?B/s]

test_data.csv:   0%|          | 0.00/870k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/83989 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10499 [00:00<?, ? examples/s]

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 83989
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10499
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 10499
    })
})

In [25]:
# Understanding Data
print("\nSample from training set:")
print(dataset['train'][0])
print(f"Text: {dataset['train'][100]['text']}")
print(f"Label: {dataset['train'][100]['label']}")


Sample from training set:
{'text': 'almost got in a giant car accident on the 101', 'label': 0}
Text: with terrific computer graphics , inventive action sequences and a droll sense of humor 
Label: 2


In [31]:
# Create mapping between labels and their names
id2label = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
label2id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}

print("Label mappings:")
for label_id, label_name in id2label.items():
    print(f"{label_id}: {label_name}")

Label mappings:
0: NEGATIVE
1: NEUTRAL
2: POSITIVE


In [26]:
# Data Tokenization
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True,padding=False, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("\nTokenized dataset structure:")
print(tokenized_dataset)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

Map:   0%|          | 0/83989 [00:00<?, ? examples/s]

Map:   0%|          | 0/10499 [00:00<?, ? examples/s]

Map:   0%|          | 0/10499 [00:00<?, ? examples/s]


Tokenized dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 83989
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10499
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10499
    })
})
Tokenizer vocab size: 30522


In [29]:
# Splitting data into training, validation and testing.
train_dataset=tokenized_dataset['train']
test_dataset=tokenized_dataset['test']
validation_dataset=tokenized_dataset["validation"]

# Will rename the label column to labels as the Trainer expects labels not label
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")
validation_dataset = validation_dataset.rename_column("label", "labels")

# Viewing the samples and length
print(f"Train: {len(train_dataset)} samples")
print(f"Validation: {len(validation_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

Train: 83989 samples
Validation: 10499 samples
Test: 10499 samples


In [32]:
# Load the model for sequence classification with 3 labels
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,  # 3 classes: negative, neutral, positive
    id2label=id2label,
    label2id=label2id
)

# Check model configuration
print(f"Model loaded with {model.config.num_labels} output labels")

NameError: name 'model_name' is not defined